## Import Library yang digunakan

In [2]:
import pandas as pd    
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import time

from scipy.special import beta as beta_func
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import fisk, burr, kstest, anderson, chisquare

## Load dan Persiapan Data

Pada bagian ini dilakukan proses pembacaan data harga saham serta tahap awal pembersihan data sebelum dilakukan analisis lebih lanjut.

In [17]:
# Load data dan print data sebelum cleaning
df_saham = pd.read_csv("lq45_mar26_close.csv", index_col="Date", parse_dates=True)
df_saham.columns = df_saham.columns.str.replace('.JK', '', regex=False)
print(df_saham.head())

            AADI  ADMR        ADRO        AKRA  AMMN        AMRT        ANTM  \
Date                                                                           
2017-01-02   NaN   NaN  422.978363  744.446777   NaN  569.011597  721.587830   
2017-01-03   NaN   NaN  434.207947  722.733643   NaN  564.459595  713.525452   
2017-01-04   NaN   NaN  424.226166  735.141174   NaN  528.042847  697.400513   
2017-01-05   NaN   NaN  415.492004  725.835632   NaN  528.042847  701.431763   
2017-01-06   NaN   NaN  422.978363  732.039124   NaN  537.147095  697.400513   

                   ASII         BBCA         BBNI  ...  NCKL         PGAS  \
Date                                               ...                      
2017-01-02  5174.229492  2480.385498  1858.162354  ...   NaN  1641.824463   
2017-01-03  5127.333496  2524.392090  1841.346436  ...   NaN  1714.794067   
2017-01-04  5002.276855  2512.390381  1883.386475  ...   NaN  1745.198486   
2017-01-05  5080.437012  2508.389893  1891.794434  ...

### Menentukan Periode Data yang Konsisten

Setiap saham memiliki tanggal awal pencatatan yang berbeda. Oleh karena itu, dilakukan pencarian tanggal pertama yang valid untuk seluruh saham.

Tujuannya adalah agar analisis hanya menggunakan periode waktu di mana semua saham memiliki data lengkap.

In [18]:
# Cari saham yang paling baryu
all_first_dates = df_saham.apply(lambda col: col.first_valid_index())
first_valid_date = all_first_dates.max()
saham_terbaru = all_first_dates.idxmax()

print(saham_terbaru)
print(first_valid_date)

AADI
2024-12-05 00:00:00


### Pembersihan Data

Dataset disesuaikan agar hanya mencakup periode waktu yang sama untuk seluruh saham.

Langkah yang dilakukan:
- Memotong data berdasarkan tanggal awal bersama
- Menghapus nilai yang mengandung missing value (NaN)

Hal ini penting untuk menjaga konsistensi dalam perhitungan return dan kovarian.

In [19]:
# Slice semua data
df_saham = df_saham.loc[first_valid_date:]

# Hapus NaN
df_saham = df_saham.dropna(axis=0, how='any')

print(f'Shape data: {df_saham.shape}')
print(f'NaN: {df_saham.isna().sum().sum()}')
print(f'Range dataset: {df_saham.index.min()} s/d {df_saham.index.max()}')

# Print data setelah cleaning
print(df_saham.head())

Shape data: (306, 45)
NaN: 0
Range dataset: 2024-12-05 00:00:00 s/d 2026-03-30 00:00:00
                   AADI         ADMR         ADRO         AKRA    AMMN  \
Date                                                                     
2024-12-05  6241.059082  1258.115356  1911.487671  1215.968994  9550.0   
2024-12-06  7484.578125  1253.349731  1878.812622  1220.592529  9525.0   
2024-12-09  8962.723633  1281.943237  2189.225098  1243.709595  9325.0   
2024-12-10  9643.139648  1258.115356  2230.068848  1252.956665  9375.0   
2024-12-11  9009.649414  1239.052979  2132.043945  1234.462769  9225.0   

                   AMRT         ANTM         ASII         BBCA         BBNI  \
Date                                                                          
2024-12-05  2951.072998  1419.330811  4788.159180  9464.409180  4071.646729   
2024-12-06  3030.031494  1414.567993  4742.119141  9302.821289  4105.436523   
2024-12-09  2901.723877  1419.330811  4811.178711  9556.745117  4223.699707  

### Visualisasi Harga Saham

Dilakukan visualisasi harga penutupan masing-masing saham dalam bentuk grafik time series.

In [ ]:
fig, axes = plt.subplots(nrows=len(df_saham.columns), ncols=1, figsize=(12, 6 * len(df_saham.columns)))

for i, stock_code in enumerate(df_saham.columns):
    axes[i].plot(df_saham.index, df_saham[stock_code], label=stock_code)
    axes[i].set_title(f"Grafik Harga Close Saham {stock_code}")
    axes[i].set_xlabel("Tanggal")
    axes[i].set_ylabel("Harga Close Saham")

    axes[i].xaxis.set_major_locator(mdates.MonthLocator())
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(axes[i].get_xticklabels(), rotation=45, ha="right")
    axes[i].legend()
    axes[i].grid(True)

plt.tight_layout()
plt.show()

### Perhitungan Log Return

Return saham dihitung menggunakan metode log return, yaitu:

$r_t = \ln \left(\frac{P_t}{P_{t-1}}\right)$.

Nilai NaN yang muncul akibat pergeseran data kemudian dihapus.

In [20]:
# Hitung log return
log_returns = np.log(df_saham / df_saham.shift(1))
print(log_returns)

log_returns.dropna().reset_index().to_csv("lq45_log_returns.csv", index=False)



                AADI      ADMR      ADRO      AKRA      AMMN      AMRT  \
Date                                                                     
2024-12-05       NaN       NaN       NaN       NaN       NaN       NaN   
2024-12-06  0.181695 -0.003795 -0.017242  0.003795 -0.002621  0.026404   
2024-12-09  0.180230  0.022557  0.152908  0.018762 -0.021221 -0.043268   
2024-12-10  0.073173 -0.018762  0.018485  0.007408  0.005348  0.036732   
2024-12-11 -0.067951 -0.015267 -0.044951 -0.014870 -0.016129 -0.026580   
...              ...       ...       ...       ...       ...       ...   
2026-03-17  0.014389  0.046884  0.004090 -0.015687  0.026260  0.017452   
2026-03-25  0.062304  0.022642  0.070896  0.050107  0.076881  0.053885   
2026-03-26 -0.018059 -0.032873 -0.019194  0.000000 -0.020203 -0.043558   
2026-03-27 -0.023042  0.012772 -0.015625  0.018623 -0.010257 -0.003431   
2026-03-30  0.084872  0.025064  0.031010  0.050370  0.006167 -0.034969   

                ANTM      ASII      B

### Visualisasi Log Return

Setiap saham divisualisasikan dalam bentuk grafik log return.

In [ ]:
for column in log_returns.columns:
    plt.figure(figsize=(10, 6))
    
    plt.plot(log_returns.index, log_returns[column])
    plt.title(f'Grafik Return Saham {column}')
    plt.xlabel('Tanggal')
    plt.ylabel('Return')
    plt.grid(True)
    plt.show()

### Statistik Deskriptif Log Return

Dilakukan analisis statistik deskriptif terhadap log return, meliputi:
- mean
- standar deviasi
- nilai minimum dan maksimum

Analisis ini memberikan gambaran awal karakteristik distribusi return masing-masing saham.

In [31]:
log_returns.describe().T

,count,mean,std,min,25%,50%,75%,max
AADI,305.0,0.002053,0.033069,-0.126441,-0.014337,-0.002837,0.016314,0.181695
ADMR,305.0,0.001552,0.037866,-0.130362,-0.020101,0.000000,0.018265,0.222238
ADRO,305.0,0.001034,0.027882,-0.114739,-0.016261,0.000000,0.012685,0.152908
AKRA,305.0,0.000520,0.029397,-0.121214,-0.013668,0.000000,0.015504,0.200671
AMMN,305.0,-0.002201,0.041816,-0.160200,-0.021221,0.000000,0.020928,0.149898
AMRT,305.0,-0.002433,0.028113,-0.093434,-0.017938,-0.003431,0.012685,0.101003
ANTM,305.0,0.002959,0.035422,-0.155171,-0.016475,0.003221,0.023690,0.109699
ASII,305.0,0.000926,0.022714,-0.097374,-0.011385,0.000000,0.011929,0.129212
BBCA,305.0,-0.001257,0.017939,-0.089153,-0.011940,-0.002684,0.008253,0.073427
BBNI,305.0,-0.000184,0.022371,-0.079022,-0.012210,-0.002318,0.008265,0.085942


### Distribusi Return Saham

Histogram log return ditampilkan untuk setiap saham.

Dua jenis histogram digunakan:
- density (probability density)
- frekuensi (count)

In [ ]:
for col in log_returns.columns:
    plt.figure()
    log_returns[col].hist(bins=50, density=True) 
    plt.title(f'Histogram (PDF) {col}')
    plt.xlabel('Return')
    plt.ylabel('Density')
    plt.show()

In [ ]:
for col in log_returns.columns:
    plt.figure()
    log_returns[col].hist(bins=50) 
    plt.title(f'Histogram {col}')
    plt.xlabel('Return')
    plt.ylabel('Frekuensi')
    plt.show()

## Estimasi Parameter Distribusi

Setelah distribusi return diidentifikasi melalui histogram, dilakukan estimasi parameter menggunakan metode **Maximum Likelihood Estimation (MLE)**.

Secara umum:

$L(\theta|x) = \prod_{i=1}^{n} f(x_i|\theta), \quad 
\ln L(\theta|x) = \sum_{i=1}^{n} \ln f(x_i|\theta)$

Contoh:

**Log-Logistik:**

$\ln L = \sum \left[ \ln \left(\frac{\alpha}{\beta}\right) + (\alpha-1)\ln \left(\frac{x_i-\gamma}{\beta}\right) - 2\ln \left(1+\left(\frac{x_i-\gamma}{\beta}\right)^\alpha\right) \right]$

**Burr (4P):**

$\ln L = \sum \left[ \ln \left(\frac{\alpha k}{\beta}\right) + (\alpha-1)\ln \left(\frac{x_i-\gamma}{\beta}\right) + (1-k)\ln \left(1+\left(\frac{x_i-\gamma}{\beta}\right)^\alpha\right) \right]$

Parameter diperoleh dengan menyelesaikan:

$\frac{\partial \ln L}{\partial \theta} = 0$

Karena tidak memiliki solusi analitik, estimasi dilakukan menggunakan software **EasyFit**.

In [6]:
df_param = pd.read_csv("nilai_parameter.csv")
print(df_param)

    No Saham         Model        k         alpha          beta         gamma
0    1  AADI     Burr (4P)  0.56587  1.027900e+02  1.279800e+00 -1.291700e+00
1    2  ADMR     Burr (4P)  0.57559  9.049300e+01  1.323200e+00 -1.337400e+00
2    3  ADRO  Log-Logistic      NaN  1.530500e+01  2.098300e-01 -2.110700e-01
3    4  AKRA     Burr (4P)  0.79324  3.199900e+02  4.263400e+00 -4.268000e+00
4    5  AMMN  Log-Logistic      NaN  2.040000e+08  4.450000e+06 -4.450000e+06
5    6  AMRT  Log-Logistic      NaN  5.143800e+01  7.813800e-01 -7.843700e-01
6    7  ANTM  Log-Logistic      NaN  7.850000e+06  1.510000e+05 -1.510000e+05
7    8  ASII     Burr (4P)  0.75371  1.020000e+08  1.040000e+06 -1.040000e+06
8    9  BBCA     Burr (4P)  0.75580  1.090000e+06  9.442700e+03 -9.442700e+03
9   10  BBNI     Burr (4P)  0.57290  2.943800e+04  2.682800e+02 -2.682900e+02
10  11  BBRI     Burr (4P)  0.65518  1.220000e+06  1.135800e+04 -1.135800e+04
11  12  BBTN     Burr (4P)  0.51255  1.740000e+07  1.790000e+05 

## Uji Kesesuaian Distribusi

Model distribusi yang diperoleh kemudian diuji menggunakan:
- Kolmogorov-Smirnov (KS)
- Anderson-Darling (AD)
- Chi-Square (CS)

Hipotesis:

$H_0$: data mengikuti distribusi  
$H_1$: data tidak mengikuti distribusi  

Kriteria:

$A_{hitung} < A_{tabel} \Rightarrow H_0$ diterima  

Distribusi dianggap sesuai jika semua uji memenuhi kriteria.

Hasil estimasi parameter dan uji distribusi selanjutnya disajikan pada tabel berikut.

In [7]:
df_uji = pd.read_csv("asumsi_model.csv")
print(df_uji)

    Saham      Uji Distribusi Model Distribusi   A Tabel  A Hitung  \
0    AADI  Kolmogorov-Smirnov        Burr (4P)   0.07776   0.06871   
1    AADI    Anderson Darling        Burr (4P)   2.50180   1.33340   
2    AADI          Chi-Square        Burr (4P)  15.50700  13.91600   
3    ADMR  Kolmogorov-Smirnov        Burr (4P)   0.07776   0.04707   
4    ADMR    Anderson Darling        Burr (4P)   2.50180   0.72919   
..    ...                 ...              ...       ...       ...   
130  UNTR    Anderson Darling        Burr (4P)   2.50180   1.40520   
131  UNTR          Chi-Square        Burr (4P)  15.50700  17.56400   
132  UNVR  Kolmogorov-Smirnov        Burr (4P)   0.07776   0.06692   
133  UNVR    Anderson Darling        Burr (4P)   2.50180   0.61636   
134  UNVR          Chi-Square        Burr (4P)  15.50700  12.35400   

             Kesimpulan Hipotesis  
0    A Hitung < A Tabel  Diterima  
1    A Hitung < A Tabel  Diterima  
2    A Hitung < A Tabel  Diterima  
3    A Hitung <

## Seleksi Saham

Dilakukan seleksi saham berdasarkan hasil uji distribusi (KS, AD, CS).

Kriteria:
- Saham dipilih jika seluruh uji menerima $H_0$

Selanjutnya, model distribusi dan parameter untuk saham terpilih digabungkan sebagai input analisis berikutnya.

In [9]:
df_uji['Saham'] = df_uji['Saham'].str.strip()
df_uji['Hipotesis'] = df_uji['Hipotesis'].str.strip()
df_param['Saham'] = df_param['Saham'].str.strip()

summary = df_uji.groupby('Saham')['Hipotesis'].apply(lambda x: (x == 'Diterima').sum())
saham_diterima = summary[summary == 3].index

df_model = df_uji[df_uji['Saham'].isin(saham_diterima)]
df_model = df_model[['Saham', 'Model Distribusi']].drop_duplicates()

df_final = pd.merge(df_model, df_param, on='Saham', how='inner')

print(f"Jumlah saham lolos semua uji: {len(saham_diterima)}")
print(list(saham_diterima))

Jumlah saham lolos semua uji: 32
['AADI', 'ADMR', 'ADRO', 'AKRA', 'AMMN', 'AMRT', 'ANTM', 'ASII', 'BBRI', 'BMRI', 'BRPT', 'BUMI', 'CPIN', 'CTRA', 'EMTK', 'ICBP', 'INCO', 'INDF', 'INKP', 'ISAT', 'JPFA', 'KLBF', 'MAPI', 'MDKA', 'MEDC', 'NCKL', 'PGAS', 'PGEO', 'SCMA', 'SMGR', 'TLKM', 'UNVR']


## Data Hasil Seleksi

Data ini merupakan hasil seleksi dan akan digunakan pada tahap perhitungan selanjutnya.

In [ ]:
df_final.head(32)

,Saham,Model Distribusi,No,Model,k,alpha,beta,gamma
0,AADI,Burr (4P),1,Burr (4P),0.56587,1.027900e+02,1.279800e+00,-1.291700e+00
1,ADMR,Burr (4P),2,Burr (4P),0.57559,9.049300e+01,1.323200e+00,-1.337400e+00
2,ADRO,Log-Logistic (3P),3,Log-Logistic,NaN,1.530500e+01,2.098300e-01,-2.110700e-01
3,AKRA,Burr (4P),4,Burr (4P),0.79324,3.199900e+02,4.263400e+00,-4.268000e+00
4,AMMN,Log-Logistic (3P),5,Log-Logistic,NaN,2.040000e+08,4.450000e+06,-4.450000e+06
5,AMRT,Log-Logistic (3P),6,Log-Logistic,NaN,5.143800e+01,7.813800e-01,-7.843700e-01
6,ANTM,Log-Logistic (3P),7,Log-Logistic,NaN,7.850000e+06,1.510000e+05,-1.510000e+05
7,ASII,Burr (4P),8,Burr (4P),0.75371,1.020000e+08,1.040000e+06,-1.040000e+06
8,BBRI,Burr (4P),11,Burr (4P),0.65518,1.220000e+06,1.135800e+04,-1.135800e+04
9,BMRI,Burr (4P),13,Burr (4P),0.85464,2.562100e+04,2.920400e+02,-2.920400e+02


## Perhitungan Expected Return dan Variansi

Pada tahap ini dihitung expected return dan variansi berdasarkan parameter distribusi yang diperoleh.

### 1. Distribusi Log-Logistik

Expected return:
$E(X) = \gamma + \beta \, B\left(1 + \frac{1}{\alpha}, 1 - \frac{1}{\alpha}\right)$

Variansi:
$\text{Var}(X) = \beta^2 \left[ B\left(1 + \frac{2}{\alpha}, 1 - \frac{2}{\alpha}\right) - \left(B\left(1 + \frac{1}{\alpha}, 1 - \frac{1}{\alpha}\right)\right)^2 \right]$


### 2. Distribusi Burr

Expected return:
$E(X) = \gamma + k\beta \, B\left(1 + \frac{1}{\alpha}, k - \frac{1}{\alpha}\right)$

Variansi:
$\text{Var}(X) = k\beta^2 B\left(1 + \frac{2}{\alpha}, k - \frac{2}{\alpha}\right) - k^2\beta^2 \left(B\left(1 + \frac{1}{\alpha}, k - \frac{1}{\alpha}\right)\right)^2$

dengan $B(\cdot,\cdot)$ adalah fungsi Beta.

In [12]:
def expected_return_loglogistic(alpha, beta_val, gamma):
    b = beta_func(1 + 1/alpha, 1 - 1/alpha)
    return gamma + beta_val * b

def varian_return_loglogistic(alpha, beta_val):
    b1 = beta_func(1 + 2/alpha, 1 - 2/alpha)
    b2 = beta_func(1 + 1/alpha, 1 - 1/alpha)
    return (beta_val**2) * (b1 - b2**2)

def expected_return_burr(k, alpha, beta_val, gamma):
    b = beta_func(1 + 1/alpha, k - 1/alpha)
    return gamma + k * beta_val * b

def varian_return_burr(k, alpha, beta_val):
    b1 = beta_func(1 + 2/alpha, k - 2/alpha)
    b2 = beta_func(1 + 1/alpha, k - 1/alpha)
    return k * (beta_val**2) * b1 - (k**2) * (beta_val**2) * (b2**2)


## Perhitungan Expected Return dan Variansi

Dilakukan perhitungan expected return dan variansi untuk setiap saham berdasarkan model distribusi (Log-Logistik atau Burr).

Hasil disimpan dalam bentuk tabel untuk digunakan pada tahap seleksi.

In [13]:
hasil = []

for i, row in df_final.iterrows():
    
    saham = row['Saham']
    model = row['Model Distribusi']
    
    k = row['k']
    alpha = row['alpha']
    beta_val = row['beta']
    gamma = row['gamma']
    
    try:
        if 'Log-Logistic' in model:
            er = expected_return_loglogistic(alpha, beta_val, gamma)
            vr = varian_return_loglogistic(alpha, beta_val)
        else:
            er = expected_return_burr(k, alpha, beta_val, gamma)
            vr = varian_return_burr(k, alpha, beta_val)
        
        hasil.append({
            'No': len(hasil) + 1,
            'Saham': saham,
            'Expected Return': er,
            'Varian Return': vr
        })
        
    except Exception as e:
        print(f"Error processing {saham}: {e}")
        
df_expretvar = pd.DataFrame(hasil)
print(df_expretvar)

df_expretvar.to_csv("expected_return_variance.csv", index=False)

    No Saham  Expected Return  Varian Return
0    1  AADI     2.137079e-03       0.000911
1    2  ADMR     1.775892e-03       0.001236
2    3  ADRO     2.407730e-04       0.000638
3    4  AKRA     8.615312e-04       0.000709
4    5  AMMN     1.862645e-09      -0.017588
5    6  AMRT    -2.504005e-03       0.000761
6    7  ANTM     3.987225e-09       0.001225
7    8  ASII     5.090405e-03      -0.000366
8    9  BBRI     7.231042e-03       0.000416
9   10  BMRI     3.059859e-03       0.000484
10  11  BRPT     8.279418e-04       0.001652
11  12  BUMI     2.084937e-03       0.002152
12  13  CPIN    -5.411644e-04       0.000457
13  14  CTRA    -1.552420e-03       0.000563
14  15  EMTK     1.456736e-02       0.001953
15  16  ICBP    -1.664422e-03       0.000329
16  17  INCO    -9.118785e-02       0.001474
17  18  INDF     4.656613e-10       0.000000
18  19  INKP     7.570224e-03       0.011719
19  20  ISAT     9.313226e-10       0.000441
20  21  JPFA     1.220004e-03       0.000913
21  22  KL

## Seleksi Saham

Saham dipilih dengan kriteria:
- $E(R) > 0$
- $\text{Var}(R) > 0$

Hasil seleksi digunakan sebagai kandidat portofolio.

In [14]:
df_selected = df_expretvar[
    (df_expretvar['Expected Return'] > 0) &
    (df_expretvar['Varian Return'] > 0)
].copy()

# reset nomor
df_selected = df_selected.reset_index(drop=True)
df_selected['No'] = df_selected.index + 1

print(df_selected)

df_selected.to_csv("saham_terpilih.csv", index=False)

    No Saham  Expected Return  Varian Return
0    1  AADI     2.137079e-03       0.000911
1    2  ADMR     1.775892e-03       0.001236
2    3  ADRO     2.407730e-04       0.000638
3    4  AKRA     8.615312e-04       0.000709
4    5  ANTM     3.987225e-09       0.001225
5    6  BBRI     7.231042e-03       0.000416
6    7  BMRI     3.059859e-03       0.000484
7    8  BRPT     8.279418e-04       0.001652
8    9  BUMI     2.084937e-03       0.002152
9   10  EMTK     1.456736e-02       0.001953
10  11  INKP     7.570224e-03       0.011719
11  12  ISAT     9.313226e-10       0.000441
12  13  JPFA     1.220004e-03       0.000913
13  14  MAPI     6.544741e-03       0.000855
14  15  MDKA     1.660113e-03       0.001858
15  16  MEDC     1.807641e-03       0.000708
16  17  NCKL     1.170920e-02       0.001203
17  18  SCMA     1.926905e-02       0.001802
18  19  TLKM     3.345394e-03       0.000608
19  20  UNVR     1.139947e-02       0.001097


## Matriks Kovarian

Menghitung matriks kovarian dari return saham terpilih.

Matriks ini merepresentasikan hubungan risiko antar saham dalam portofolio.

In [21]:
selected_stocks = df_selected['Saham'].tolist()

# ambil log return asli
df_return_selected = log_returns[selected_stocks]

df_cov_matrix = df_return_selected.cov()
print(df_cov_matrix)

df_cov_matrix.to_csv("covariance_matrix.csv")

          AADI      ADMR      ADRO      AKRA      ANTM      BBRI      BMRI  \
AADI  0.001094  0.000557  0.000492  0.000251  0.000182  0.000132  0.000160   
ADMR  0.000557  0.001434  0.000632  0.000286  0.000490  0.000162  0.000171   
ADRO  0.000492  0.000632  0.000777  0.000171  0.000283  0.000177  0.000217   
AKRA  0.000251  0.000286  0.000171  0.000864  0.000284  0.000177  0.000179   
ANTM  0.000182  0.000490  0.000283  0.000284  0.001255  0.000077  0.000082   
BBRI  0.000132  0.000162  0.000177  0.000177  0.000077  0.000479  0.000362   
BMRI  0.000160  0.000171  0.000217  0.000179  0.000082  0.000362  0.000516   
BRPT  0.000102  0.000258  0.000181  0.000172  0.000288  0.000206  0.000199   
BUMI  0.000471  0.000487  0.000282  0.000356  0.000424  0.000142  0.000214   
EMTK  0.000253  0.000224  0.000121  0.000148  0.000218  0.000201  0.000256   
INKP  0.000233  0.000504  0.000265  0.000263  0.000373  0.000218  0.000221   
ISAT  0.000205  0.000384  0.000276  0.000179  0.000250  0.000247

## Persiapan Parameter Optimasi

Menyusun parameter utama:
- $\mu$ : vektor expected return
- $e$ : vektor satuan
- $\Sigma$ : matriks kovarian
- $\Sigma^{-1}$ : invers kovarian

Parameter ini digunakan dalam proses optimasi portofolio.

In [ ]:
mu = df_selected['Expected Return'].values.reshape(-1, 1)
e = np.ones((len(mu), 1))
sigma = df_cov_matrix.values
sigma_inv = np.linalg.inv(sigma)

In [23]:
print("mu:\n", mu)

mu:
 [[2.13707909e-03]
 [1.77589150e-03]
 [2.40773014e-04]
 [8.61531161e-04]
 [3.98722477e-09]
 [7.23104202e-03]
 [3.05985912e-03]
 [8.27941793e-04]
 [2.08493678e-03]
 [1.45673594e-02]
 [7.57022388e-03]
 [9.31322575e-10]
 [1.22000388e-03]
 [6.54474095e-03]
 [1.66011281e-03]
 [1.80764057e-03]
 [1.17092031e-02]
 [1.92690542e-02]
 [3.34539386e-03]
 [1.13994702e-02]]


In [24]:
print("\ne:\n", e)


e:
 [[1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]]


In [25]:
print("\nsigma:\n", sigma)


sigma:
 [[1.09356822e-03 5.57321089e-04 4.91585201e-04 2.50750879e-04
  1.81974962e-04 1.31549761e-04 1.59756246e-04 1.02329814e-04
  4.71382927e-04 2.52912852e-04 2.32773054e-04 2.05072062e-04
  1.17518037e-04 1.80490612e-04 2.75959102e-04 1.28475282e-04
  2.42858915e-04 3.20043829e-04 1.21714688e-04 1.70201971e-04]
 [5.57321089e-04 1.43382297e-03 6.32383427e-04 2.85980754e-04
  4.90428066e-04 1.62405563e-04 1.70817450e-04 2.58190966e-04
  4.86783030e-04 2.23913396e-04 5.04303214e-04 3.84332984e-04
  7.19241895e-05 2.61030554e-04 6.07291931e-04 2.88060912e-04
  5.39464579e-04 3.02992014e-04 2.24121956e-04 2.64071420e-04]
 [4.91585201e-04 6.32383427e-04 7.77432249e-04 1.71193443e-04
  2.83037123e-04 1.76849056e-04 2.17427572e-04 1.80883395e-04
  2.82459428e-04 1.21137006e-04 2.65068271e-04 2.75641204e-04
  9.51453253e-05 1.62138636e-04 4.02302751e-04 1.51283063e-04
  3.21127019e-04 2.00834399e-04 1.76971881e-04 1.09907302e-04]
 [2.50750879e-04 2.85980754e-04 1.71193443e-04 8.64173171e

In [26]:
print("\nsigma inverse:\n", sigma_inv)


sigma inverse:
 [[ 1.45534132e+03 -2.28555345e+02 -7.35960358e+02 -1.86965889e+02
   5.75769379e+01  2.63536094e+01 -1.55088654e+01  1.07320365e+02
  -1.54141580e+02 -8.04773631e+01  3.92815543e+01  1.02301625e+01
  -2.84106939e+01 -4.91901387e+01  6.24566280e+01 -1.47663548e+01
   7.52428640e+01 -5.75010822e+01  1.11297052e+02 -7.45139428e+01]
 [-2.28555345e+02  1.38457570e+03 -7.51900717e+02 -2.51003805e+01
  -9.35826608e+01 -5.57695004e+01  3.25743198e+02  1.18602033e+01
  -4.28618375e+01  3.01979584e+01 -2.19083213e+02 -1.42979369e+02
   1.42549461e+02 -4.60482193e+01 -2.70513203e+01 -2.05855699e+02
  -1.76496289e+02  2.32769172e+01 -3.69319864e+01 -9.64544719e+01]
 [-7.35960358e+02 -7.51900717e+02  2.65787930e+03  1.20029524e+02
  -7.59230102e+01 -1.20076802e+02 -5.95121096e+02 -6.60971830e+01
   8.59474453e+01  2.05401396e+02  5.12060002e+01 -3.35200628e+01
  -6.46545963e+01  6.23368424e+01 -1.40356604e+02  5.48311051e+01
  -1.61101323e+02 -8.49223446e+01 -1.62658034e+02  2.0728